# Phase A 

In [8]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

In [9]:
def charger_immobilier():
    data = fetch_california_housing()
    print("California Housing :", data.data.shape)
    print("Variables :", data.feature_names)
    print("Cible : prix médian en centaines de milliers de dollars")
    return data.data, data.target

In [10]:
def evaluer_regression(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    predictions = modele.predict(X_test)
    return {
        "r2": r2_score(y_test, predictions),
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": root_mean_squared_error(y_test, predictions)
    }

## Test 1 

In [11]:
X, y = charger_immobilier()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

California Housing : (20640, 8)
Variables : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Cible : prix médian en centaines de milliers de dollars


In [12]:
modeles = {
    "Régression linéaire": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

for nom, modele in modeles.items():
    scores = evaluer_regression(modele, X_train_scaled, X_test_scaled, y_train, y_test)
    print(f"{nom} : R2={scores['r2']:.2f} MAE={scores['mae']:.2f} RMSE={scores['rmse']:.2f}")

Régression linéaire : R2=0.58 MAE=0.53 RMSE=0.75
Random Forest : R2=0.80 MAE=0.33 RMSE=0.51


## Test 2 

In [13]:
X_100_train, X_100_test, y_100_train, y_100_test = train_test_split(
    X[:100], y[:100], test_size=0.2, random_state=42
)
scaler_100 = StandardScaler()
X_100_train = scaler_100.fit_transform(X_100_train)
X_100_test = scaler_100.transform(X_100_test)

scores_100 = evaluer_regression(
    LinearRegression(), X_100_train, X_100_test, y_100_train, y_100_test
)
print("Régression linéaire sur 100 lignes :", scores_100)

Régression linéaire sur 100 lignes : {'r2': 0.7120875310155536, 'mae': 0.3837282351053749, 'rmse': 0.4933452872936244}


## Test 3 

In [14]:
modele_lineaire = LinearRegression().fit(X_train_scaled, y_train)

quartier_fictif = X[0].copy()
quartier_fictif[0] = 0
quartier_fictif[4] = 9000

prediction = modele_lineaire.predict(scaler.transform([quartier_fictif]))[0]
print("Prix prédit :", prediction, "centaines de milliers de dollars")
print("En production, il faut détecter ou refuser les entrées hors plage.")

Prix prédit : 0.39902128040945684 centaines de milliers de dollars
En production, il faut détecter ou refuser les entrées hors plage.


# Phase B 

In [15]:
def charger_airbnb(url_csv):
    df = pd.read_csv(url_csv)
    colonnes = ["price", "minimum_nights", "number_of_reviews", "reviews_per_month", "availability_365"]
    df = df[colonnes]
    df["price"] = pd.to_numeric(df["price"].astype(str).str.replace("$", "").str.replace(",", ""), errors="coerce")
    df = df.dropna()
    df = df[df["price"] <= df["price"].quantile(0.99)].head(8000)
    print("Listings chargés :", df.shape)
    return df

In [16]:
def choisir_k(X_scaled, k_range=range(2, 9)):
    resultats = []
    for k in k_range:
        modele = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = modele.fit_predict(X_scaled)
        silhouette = silhouette_score(X_scaled, labels)
        resultats.append((k, modele.inertia_, silhouette))
        print(f"k={k} : inertie={modele.inertia_:.0f} silhouette={silhouette:.2f}")
    return resultats

## Test 1 

In [17]:
url_airbnb = "../data/airbnb_listings.csv"
airbnb = charger_airbnb(url_airbnb)
airbnb_scaled = StandardScaler().fit_transform(airbnb)
resultats_k = choisir_k(airbnb_scaled)
meilleur_k = max(resultats_k, key=lambda resultat: resultat[2])[0]
print("Segment retenu :", meilleur_k)

Listings chargés : (5165, 5)
k=2 : inertie=21179 silhouette=0.41
k=3 : inertie=17239 silhouette=0.36
k=4 : inertie=13702 silhouette=0.37
k=5 : inertie=10685 silhouette=0.38
k=6 : inertie=8496 silhouette=0.40
k=7 : inertie=7650 silhouette=0.40
k=8 : inertie=6952 silhouette=0.39
Segment retenu : 2


In [18]:
modele_airbnb = KMeans(n_clusters=meilleur_k, n_init=10, random_state=42)
airbnb["segment"] = modele_airbnb.fit_predict(airbnb_scaled)
print(airbnb.groupby("segment").mean().round(1))

         price  minimum_nights  number_of_reviews  reviews_per_month  \
segment                                                                
0        162.6             2.0              386.1                5.1   
1        263.8             4.0               28.7                0.8   

         availability_365  
segment                    
0                   196.6  
1                   143.7  


## Test 2 

In [19]:
colonnes_airbnb = airbnb.drop(columns="segment")
labels_sans_scale = KMeans(n_clusters=meilleur_k, n_init=10, random_state=42).fit_predict(colonnes_airbnb)
print("Silhouette sans scaling :", silhouette_score(colonnes_airbnb, labels_sans_scale))
print("Sans scaling, les colonnes avec les grandes valeurs dominent les groupes.")

Silhouette sans scaling : 0.5103292274347653
Sans scaling, les colonnes avec les grandes valeurs dominent les groupes.


## Test 3 

In [20]:
airbnb_outlier = colonnes_airbnb.copy()
annonce_extreme = airbnb_outlier.iloc[[0]].copy()
annonce_extreme["price"] = 100000
airbnb_outlier = pd.concat([airbnb_outlier, annonce_extreme], ignore_index=True)

outlier_scaled = StandardScaler().fit_transform(airbnb_outlier)
modele_outlier = KMeans(n_clusters=meilleur_k, n_init=10, random_state=42).fit(outlier_scaled)
print("Prix maximum avant :", colonnes_airbnb["price"].max())
print("Prix maximum après injection :", airbnb_outlier["price"].max())
print("Une valeur extrême déplace le scaler et les centres des clusters.")

Prix maximum avant : 948.0
Prix maximum après injection : 100000.0
Une valeur extrême déplace le scaler et les centres des clusters.
